# 🚪 Day 05a: API Gateway, Rate Limiting & Circuit Breaker

---

## 🎯 What You'll Master Today
- API Gateway pattern — why every microservice system needs one
- Reverse proxy vs API gateway
- Rate limiting algorithms
- Circuit breaker pattern
- Service discovery & service mesh

**The analogy:** An API gateway is the RECEPTION DESK of a building — all visitors go through it, get ID-checked, and directed to the right office.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  API GATEWAY — THE FRONT DOOR                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Without gateway:              With gateway:                     ║
║                                                                   ║
║  Client──►User Svc             Client──► [API Gateway]          ║
║  Client──►Order Svc                        │                     ║
║  Client──►Payment Svc                      ├──► User Svc        ║
║  Client──►Search Svc                       ├──► Order Svc       ║
║                                            ├──► Payment Svc     ║
║  Problems:                                 └──► Search Svc      ║
║  - Client knows all services                                     ║
║  - No single auth/rate-limit    Gateway handles:                ║
║  - Hard to change service URLs  ✅ Authentication / JWT         ║
║  - CORS for each service        ✅ Rate limiting                ║
║                                 ✅ Request routing              ║
║                                 ✅ Load balancing               ║
║                                 ✅ Request/Response transform   ║
║                                 ✅ SSL termination              ║
║                                 ✅ Caching                      ║
║                                 ✅ Logging/monitoring           ║
║                                 ✅ Circuit breaking             ║
║                                                                   ║
║  Tools: Kong, AWS API Gateway, Nginx, Traefik, Envoy            ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Reverse Proxy vs API Gateway
# ============================================================

print("""
REVERSE PROXY vs API GATEWAY
════════════════════════════

┌──────────────────┬─────────────────────┬─────────────────────┐
│ Feature          │ Reverse Proxy       │ API Gateway         │
├──────────────────┼─────────────────────┼─────────────────────┤
│ Core purpose     │ Forward requests to │ Manage + route APIs │
│                  │ backend servers     │                     │
│ SSL termination  │ ✅                  │ ✅                  │
│ Load balancing   │ ✅                  │ ✅                  │
│ Caching          │ ✅                  │ ✅                  │
│ Auth/AuthZ       │ Basic               │ ✅ Full JWT/OAuth   │
│ Rate limiting    │ Basic               │ ✅ Per-user/API key │
│ Request transform│ ❌                  │ ✅ Aggregate, map   │
│ API analytics    │ ❌                  │ ✅ Per-endpoint     │
│ API versioning   │ ❌                  │ ✅                  │
│ Circuit breaker  │ ❌                  │ ✅                  │
│ Example          │ Nginx, HAProxy      │ Kong, AWS APIGW     │
└──────────────────┴─────────────────────┴─────────────────────┘

💡 API Gateway = Reverse Proxy + Business Logic Layer

BFF (Backend for Frontend) pattern:
  Mobile App ──► [Mobile API Gateway] ──► microservices
  Web App    ──► [Web API Gateway]    ──► microservices
  Each gateway tailors responses for its client type.
""")

In [ ]:
# ============================================================
# 2. Rate Limiting Algorithms
# ============================================================

import time

# --- Token Bucket ---
class TokenBucket:
    """
    Allows burst up to bucket size, then steady rate.
    Used by: AWS API Gateway, Stripe
    """
    def __init__(self, rate, capacity):
        self.rate = rate          # tokens per second
        self.capacity = capacity  # max burst
        self.tokens = capacity
        self.last_refill = time.time()
    
    def allow_request(self):
        now = time.time()
        # Refill tokens
        elapsed = now - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        self.last_refill = now
        
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

# --- Sliding Window Counter ---
class SlidingWindowCounter:
    """
    Weighted count of current + previous window.
    More accurate than fixed window, less memory than log.
    """
    def __init__(self, limit, window_sec=60):
        self.limit = limit
        self.window = window_sec
        self.prev_count = 0
        self.curr_count = 0
        self.window_start = time.time()
    
    def allow_request(self):
        now = time.time()
        elapsed = now - self.window_start
        
        if elapsed >= self.window:
            self.prev_count = self.curr_count
            self.curr_count = 0
            self.window_start = now
            elapsed = 0
        
        # Weighted estimate
        weight = 1 - (elapsed / self.window)
        estimated = self.prev_count * weight + self.curr_count
        
        if estimated < self.limit:
            self.curr_count += 1
            return True
        return False

print("""
RATE LIMITING ALGORITHMS
════════════════════════

Algorithm          │ Pros                  │ Cons
───────────────────│───────────────────────│──────────────────
Token Bucket       │ Allows burst, smooth  │ Needs per-user state
Leaky Bucket       │ Constant output rate  │ No burst allowed
Fixed Window       │ Simple                │ Spike at boundary
Sliding Window Log │ Most accurate         │ High memory (stores timestamps)
Sliding Window Cnt │ Good balance          │ Approximate

💡 Token Bucket is most common in production (AWS, Stripe, GitHub)
""")

# Demo
tb = TokenBucket(rate=2, capacity=5)  # 2 req/sec, burst of 5
results = [tb.allow_request() for _ in range(8)]
print(f"Token Bucket (rate=2, burst=5): {results}")
print(f"First 5 allowed (burst), then limited")

In [ ]:
# ============================================================
# 3. Circuit Breaker Pattern
# ============================================================

import random

class CircuitBreaker:
    """
    Prevents cascading failures.
    If downstream service keeps failing, stop calling it.
    
    States: CLOSED → OPEN → HALF_OPEN → CLOSED
    """
    CLOSED = "CLOSED"       # Normal — requests flow through
    OPEN = "OPEN"           # Failing — reject immediately
    HALF_OPEN = "HALF_OPEN" # Testing — allow one request
    
    def __init__(self, failure_threshold=5, recovery_timeout=10):
        self.state = self.CLOSED
        self.failure_count = 0
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.last_failure_time = 0
    
    def call(self, func):
        if self.state == self.OPEN:
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = self.HALF_OPEN
                print("  [Circuit] HALF_OPEN — testing...")
            else:
                print("  [Circuit] OPEN — request rejected (fail fast)")
                return None
        
        try:
            result = func()
            if self.state == self.HALF_OPEN:
                self.state = self.CLOSED
                self.failure_count = 0
                print("  [Circuit] CLOSED — service recovered!")
            return result
        except Exception as e:
            self.failure_count += 1
            self.last_failure_time = time.time()
            if self.failure_count >= self.failure_threshold:
                self.state = self.OPEN
                print(f"  [Circuit] OPEN — {self.failure_count} failures!")
            raise

print("""
CIRCUIT BREAKER — STATE MACHINE
═══════════════════════════════

  ┌────────┐  failure >= threshold  ┌────────┐
  │ CLOSED │──────────────────────►│  OPEN  │
  │(normal)│                        │(reject)│
  └────────┘◄─── success ───────── └────┬───┘
       ▲                                 │
       │         ┌───────────┐          │
       └─success─│ HALF_OPEN │◄─timeout─┘
                 │ (test one)│
                 └───────────┘

Why it matters:
  Without: Service A calls Service B (down) → A times out → 
           all A's threads stuck → A goes down → cascade!
  With: After 5 failures, A stops calling B → fails fast →
        A stays healthy, periodically retests B.

Tools: Resilience4j (Java), pybreaker (Python), Hystrix (Netflix)
""")

In [ ]:
# ============================================================
# 4. Service Discovery
# ============================================================

print("""
SERVICE DISCOVERY
═════════════════

Problem: In microservices, services come and go (scaling, deployment).
         How does Service A find Service B's current address?

1️⃣ CLIENT-SIDE DISCOVERY
   Service A queries registry → gets list → picks one
   
   [Service A] ──query──► [Registry]     (Eureka, Consul)
       │                    │
       │   returns:         │
       │   B: 10.0.1.5:8080│
       │   B: 10.0.1.6:8080│
       └──direct call──► [Service B @ 10.0.1.5]

2️⃣ SERVER-SIDE DISCOVERY
   Service A calls load balancer → LB queries registry → routes
   
   [Service A] ──► [LB] ──query──► [Registry]
                    │
                    └──route──► [Service B]
   
   This is what AWS ALB + ECS/EKS does.

3️⃣ DNS-BASED
   service-b.internal:8080 → DNS resolves to current IPs
   Kubernetes uses this! (service-name.namespace.svc.cluster.local)


SERVICE MESH (Advanced)
══════════════════════

  [Svc A] ──► [Sidecar Proxy] ──► [Sidecar Proxy] ──► [Svc B]
                (Envoy)               (Envoy)
                    │                     │
                    └── Control Plane ────┘
                        (Istio / Linkerd)

  Sidecar handles: mTLS, retries, circuit breaking, observability
  App code is clean — no networking logic!

  Use when: 50+ microservices, need mTLS everywhere,
            consistent observability across all services.
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Why API gateway? | Single entry point, auth, rate limit, routing, SSL termination, monitoring |
| 2 | Rate limiting algorithm? | Token bucket (most common). Allows burst, then steady rate. Per-user with Redis |
| 3 | What is circuit breaker? | Stops calling failing services. CLOSED→OPEN→HALF_OPEN. Prevents cascading failure |
| 4 | Client-side vs server-side discovery? | Client: queries registry directly. Server: LB handles it. Kubernetes uses DNS |
| 5 | Reverse proxy vs API gateway? | RP = forward+SSL+LB. Gateway = RP + auth + rate limit + transform + analytics |
| 6 | What is a service mesh? | Sidecar proxies handle networking (mTLS, retries, tracing). Istio, Linkerd |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • API Gateway = single entry for all microservices      │
## │  • Token Bucket is the go-to rate limiter                │
## │  • Circuit Breaker prevents cascading failures           │
## │  • Service discovery: DNS (K8s) or registry (Consul)     │
## │  • Service mesh for 50+ services (Istio + Envoy)         │
## │  • BFF pattern: separate gateway per client type         │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **LLD — Parking Lot System Design**